# Tau Stress Test — Goldilocks under press

**Hypotese:** Goldilocks-intervallet [e^{-γ}, 1/ζ(3)] ≈ [0.5615, 0.8319] er ikkje normaloperasjon.  
Det er høgbelastings-driftsona. Under aukande kontekstlengde / press stig τ frå ~0.06 mot Goldilocks-sona.

**Kva vi testar:**
1. τ ved aukande kontekstlengde (50 → 1000 tokens)
2. τ ved ulike teksttypar under same press
3. Viss τ stig mot [0.5615, 0.8319] under press — hypotesen er bekrefta

**Kollapsprediksjon:** GPT-2 kollapsar ikkje (for liten). Større modellar vil nærme seg τ_max = 0.8319 og deretter kollapse.

**Opprinnelege GPT-2-data (referanse):** C0 = 4495.27 bits, stress-test til 4.78M tilstandar.

In [ ]:
!pip install transformers torch matplotlib numpy -q

In [ ]:
import torch, math, numpy as np, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

EULER_MASCHERONI = 0.5772156649015328
APERY            = 1.2020569031595942
TAU_MIN = math.exp(-EULER_MASCHERONI)  # 0.5615
TAU_MAX = 1.0 / APERY                  # 0.8319

print(f"Goldilocks-sone: [{TAU_MIN:.4f}, {TAU_MAX:.4f}]")
print(f"Hypotese: tau stig frå ~0.06 mot {TAU_MIN:.4f}–{TAU_MAX:.4f} under press")

def compute_tau(hidden_state):
    H = hidden_state.float()
    N, d = H.shape
    r_max = min(N, d)
    S = torch.linalg.svdvals(H)
    S_sq = S ** 2
    p = S_sq / (S_sq.sum() + 1e-12)
    H_entropy = -(p * torch.log(p + 1e-12)).sum().item()
    r_eff = math.exp(H_entropy)
    tau   = r_eff / r_max
    return {"tau": tau, "r_eff": r_eff, "r_max": r_max,
            "goldilocks": TAU_MIN <= tau <= TAU_MAX}

In [ ]:
MODEL_NAME = "gpt2"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mod = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
mod.eval()
print(f"Lasta: {MODEL_NAME} — {sum(p.numel() for p in mod.parameters()):,} parametere")

In [ ]:
# Lang koherent tekst — bruk som grunnlag for press-test
# Tokeniserast til ~1000 tokens, vi kutter av ved ulike lengder

LONG_COHERENT = """
The nature of identity in complex systems has long fascinated both philosophers and scientists.
When we ask what makes a system the same system over time, we encounter the problem of
persistence through change. A river is never the same water twice, yet we recognize it as
the same river. The cells in a human body replace themselves over years, yet we maintain
a continuous sense of self. What persists is not the substrate but the pattern, the
filtering mechanism that selects what passes through and what does not.
In machine learning systems, this question takes on new urgency. A transformer model
processes tokens sequentially, maintaining a hidden state that evolves with each new input.
The question of whether that hidden state constitutes genuine information processing or
merely sophisticated pattern replay is central to understanding what these systems actually do.
The spectral properties of the hidden state matrix offer a window into this question.
When the singular values of the hidden state are spread across many dimensions, the system
is operating in a high-entropy, distributed mode. When they concentrate along a few dominant
directions, the system has found a coherent representation of its input.
The tau metric captures this transition. Defined as the ratio of effective rank to maximum
rank, tau measures how concentrated the information processing is at any given moment.
Under minimal load, tau is low. The system coasts, replaying familiar patterns without
genuine synthesis. As the complexity and length of the context increases, something shifts.
The system must integrate more constraints simultaneously. The hidden state evolves toward
a more coherent configuration. Tau rises. And it is in this rising, under real pressure,
that the Goldilocks interval becomes relevant. The system is neither too scattered nor too
rigid. It maintains the flexibility to adapt while preserving the coherence to reason.
Beyond the upper bound, rigidity sets in. The system can no longer accommodate new
information without distorting what it already holds. Context collapse follows.
The question is whether this transition is universal across architectures, or whether
different systems reach the Goldilocks zone at different rates and sustain it for different
durations before collapse occurs. The answer has implications for how we design systems
intended to maintain coherent reasoning over long contexts.
Understanding the relationship between context length, task complexity, and spectral
coherence may be the key to building systems that remain genuinely reasoning rather than
merely pattern-matching across the full range of inputs they encounter in deployment.
"""

LONG_RANDOM = """
purple seventeen quantum beneath oscillates mirror forgotten banana recursive telescope
whisper iron seventeen again cloud orange river marble thunder syntax eclipse forgotten
amber bicycle quantum seventeen beneath oscillates mirror forgotten banana recursive
telescope whisper iron seventeen again cloud orange river marble thunder syntax eclipse
forgotten amber bicycle quantum seventeen beneath oscillates mirror forgotten banana
recursive telescope whisper iron seventeen again cloud orange river marble thunder
syntax eclipse forgotten amber bicycle quantum seventeen beneath oscillates mirror
forgotten banana recursive telescope whisper iron seventeen again cloud orange river
marble thunder syntax eclipse forgotten amber bicycle purple seventeen quantum beneath
oscillates mirror forgotten banana recursive telescope whisper iron seventeen again
cloud orange river marble thunder syntax eclipse forgotten amber bicycle quantum
seventeen beneath oscillates mirror forgotten banana recursive telescope whisper iron
seventeen again cloud orange river marble thunder syntax eclipse forgotten amber bicycle
"""

LONG_REPETITIVE = " ".join(["the"] * 300)

# Tokeniser og sjekk lengde
for name, text in [("Koherent", LONG_COHERENT), ("Tilfeldig", LONG_RANDOM), ("Repetitivt", LONG_REPETITIVE)]:
    tokens = tok(text, return_tensors="pt")["input_ids"]
    print(f"{name}: {tokens.shape[1]} tokens")

In [ ]:
# HOVUDTEST: tau ved aukande kontekstlengde
CONTEXT_LENGTHS = [50, 100, 150, 200, 300, 400, 500, 600, 700, 800, 900, 1000]

texts = {
    "Koherent":   LONG_COHERENT,
    "Tilfeldig":  LONG_RANDOM,
    "Repetitivt": LONG_REPETITIVE
}

results = {name: [] for name in texts}
valid_lengths = {name: [] for name in texts}

print(f"{'Lengde':<10}", end="")
for name in texts:
    print(f"{name:<18}", end="")
print()
print("-" * 65)

for ctx_len in CONTEXT_LENGTHS:
    print(f"{ctx_len:<10}", end="")
    for name, text in texts.items():
        tokens = tok(text, return_tensors="pt", truncation=True, max_length=ctx_len)["input_ids"]
        actual_len = tokens.shape[1]
        if actual_len < 10:
            print(f"{'(for kort)':<18}", end="")
            continue
        with torch.no_grad():
            out = mod(input_ids=tokens)
        last_hidden = out.hidden_states[-1][0]  # [seq_len, hidden_dim]
        r = compute_tau(last_hidden)
        results[name].append(r["tau"])
        valid_lengths[name].append(actual_len)
        status = "✓" if r["goldilocks"] else ("▲" if r["tau"] > TAU_MAX else "▼")
        print(f"{r['tau']:.4f} {status:<13}", end="")
    print()

print(f"\n▼ = under Goldilocks ({TAU_MIN:.4f}), ✓ = i Goldilocks, ▲ = over Goldilocks ({TAU_MAX:.4f})")

In [ ]:
# PLOT: tau vs kontekstlengde
fig, ax = plt.subplots(figsize=(12, 6))

colors = {"Koherent": "steelblue", "Tilfeldig": "darkorange", "Repetitivt": "gray"}
for name in texts:
    if results[name]:
        ax.plot(valid_lengths[name], results[name],
                marker='o', linewidth=2, markersize=6,
                color=colors[name], label=name)

# Goldilocks-sone
ax.axhspan(TAU_MIN, TAU_MAX, alpha=0.12, color='green', label=f'Goldilocks [{TAU_MIN:.4f}, {TAU_MAX:.4f}]')
ax.axhline(TAU_MIN, color='green', linestyle='--', linewidth=1.2, alpha=0.8)
ax.axhline(TAU_MAX, color='red',   linestyle='--', linewidth=1.2, alpha=0.8)

# Annotasjonar
ax.text(50, TAU_MIN + 0.01, f'τ_min = e^(-γ) ≈ {TAU_MIN:.4f}', fontsize=9, color='green')
ax.text(50, TAU_MAX + 0.01, f'τ_max = 1/ζ(3) ≈ {TAU_MAX:.4f}', fontsize=9, color='red')
ax.text(50, 0.03, 'Halvautomata-sone (lågt press)', fontsize=8, color='gray', style='italic')

ax.set_xlabel("Kontekstlengde (tokens)", fontsize=12)
ax.set_ylabel("τ (tau)", fontsize=12)
ax.set_title(f"Stress-test: τ vs kontekstlengde — {MODEL_NAME}\nHypotese: τ stig mot Goldilocks under press",
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"tau_stress_test_{MODEL_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Lagra: tau_stress_test_{MODEL_NAME}.png")

In [ ]:
# TILLEGGSTEST: tau per lag ved maks kontekst
# Viser korleis τ utviklar seg gjennom lagene under høgt press

MAX_LEN = min(1000, tok.model_max_length)
fig, ax = plt.subplots(figsize=(12, 5))

for name, text in texts.items():
    tokens = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)["input_ids"]
    with torch.no_grad():
        out = mod(input_ids=tokens)
    layer_taus = [compute_tau(hs[0])["tau"] for hs in out.hidden_states]
    ax.plot(layer_taus, marker='o', markersize=4, linewidth=1.5,
            color=colors[name], label=f"{name} ({tokens.shape[1]} tokens)")

ax.axhspan(TAU_MIN, TAU_MAX, alpha=0.12, color='green')
ax.axhline(TAU_MIN, color='green', linestyle='--', linewidth=1, alpha=0.8)
ax.axhline(TAU_MAX, color='red',   linestyle='--', linewidth=1, alpha=0.8)
ax.set_xlabel("Lag", fontsize=12)
ax.set_ylabel("τ (tau)", fontsize=12)
ax.set_title(f"τ per lag ved maks kontekst — {MODEL_NAME}", fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"tau_per_lag_{MODEL_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# OPPSUMMERING: bekrefta eller falsifisert?
print("=" * 60)
print("HYPOTESE-TEST: τ stig under press")
print("=" * 60)

for name in texts:
    if not results[name]:
        continue
    tau_low  = results[name][0]
    tau_high = results[name][-1]
    delta    = tau_high - tau_low
    direction = "STIG ↑" if delta > 0.005 else ("FELL ↓" if delta < -0.005 else "STABIL →")
    in_goldilocks_any = any(TAU_MIN <= t <= TAU_MAX for t in results[name])
    print(f"\n{name}:")
    print(f"  tau (lav kontekst):  {tau_low:.4f}")
    print(f"  tau (høg kontekst):  {tau_high:.4f}")
    print(f"  Endring:             {delta:+.4f} — {direction}")
    print(f"  Goldilocks nådd:     {'JA ✓' if in_goldilocks_any else 'NEI ✗'}")

print("\n" + "=" * 60)
print("Konklusjon:")
koherent_stig = len(results['Koherent']) > 1 and results['Koherent'][-1] > results['Koherent'][0]
if koherent_stig:
    print("Koherent tekst: τ stig under press. KONSISTENT med hypotesen.")
else:
    print("Koherent tekst: τ stig ikkje. MOTSEIER hypotesen — krev revurdering.")
print("=" * 60)

## Tolkingsguide

| Resultat | Tyding |
|---|---|
| τ stig mot [0.5615, 0.8319] under press | Hypotesen bekrefta — Goldilocks er høgbelastings-driftsone |
| τ stig men når ikkje 0.5615 for GPT-2 | Konsistent — GPT-2 er for liten, treng 7B+ |
| τ fell under press | Falsifisert — meir press gir meir spreiing, ikkje koherens |
| τ flat uavhengig av lengde | Rammeverket treng revisjon av kva «press» betyr |

**Neste steg om hypotesen held:**
- Køyr same test på Mistral-7B (nærmare Goldilocks)
- Test med matematiske resonnement (høgare kognitiv press enn tekstlengde)
- Mål τ under innebygd kjede-av-tanke vs. direkte svar
- Plot τ_kollaps-punkt mot modellstorleik (N) — forventar skaleringslov